<a href="https://colab.research.google.com/github/ozodbekAI/Data-Scince-and-AI-Portfolio/blob/main/Deepseek_fine_tune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --upgrade transformers
!pip install -U transformers datasets accelerate peft bitsandbytes trl

In [ ]:
from datasets import load_dataset

dataset = load_dataset("UAzimov/uzbek-instruct-llm")

print(dataset)
print(dataset["train"][0])

In [ ]:
print(len(dataset["train"]))
print(dataset["train"].column_names)

In [ ]:
print(dataset['train'][0])

In [ ]:
sample = dataset["train"][0]

for message in sample["messages"]:
  print(f"Role: {message['role']}")
  print(f"Content: {message["content"]}")


In [ ]:
from collections import Counter

role_counter = Counter()

for sample in dataset["train"]:
  for message in sample["messages"]:
    role_counter[message["role"]] += 1

print(role_counter)

In [ ]:
system_lenghts = []

for sample in dataset["train"]:
  for message in sample["messages"]:
    if message["role"] == "system":
      system_lenghts.append(len(message["content"]))

print(f"Min: {min(system_lenghts)}")
print(f"Max: {max(system_lenghts)}")
print(f"Average: {sum(system_lenghts) / len(system_lenghts)}")
print(f"System samples: {len(system_lenghts)}")

In [ ]:
import pandas as pd

df = dataset["train"].to_pandas()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.head()

In [ ]:
df_messages = df.explode("messages", ignore_index=True)

messages_df = pd.json_normalize(df_messages["messages"])

messages_df.head()

In [ ]:
print("Samples:", len(df))
print("Messages:", len(messages_df))
print("\nRoles:")
print(messages_df["role"].value_counts())

In [ ]:
messages_df["char_length"] = messages_df["content"].str.len()

print(
    messages_df.groupby("role")["char_length"]
    .describe()
    .round(2)
)

In [ ]:
import matplotlib.pyplot as plt

for role in messages_df["role"].unique():
  messages_df.loc[
      messages_df["role"] == role,
      "char_length"
  ].plot(
      kind="hist",
      bins=50,
      figsize=(10, 5),
      title=f"Distribution of {role} messages",
      alpha=0.5
  )

  plt.xlabel("Character length")
  plt.ylabel("Frequency")
  plt.show()

In [ ]:
messages_df.nlargest(
    10,
    "char_length"
)[["role", "char_length", "content"]]

In [ ]:
print(messages_df.isna().sum())

In [ ]:
from datasets import Dataset

full_dataset = Dataset.from_pandas(
    df[["messages"]],
    preserve_index=False
)

print(full_dataset)

In [ ]:
# Split the dataset into 80% for training and 20% for a temporary test/validation set
train_dataset, temp_test_val = full_dataset.train_test_split(test_size=0.2, seed=42).values()

# Split the temporary test/validation set into 10% validation and 10% test
validation_dataset, test_dataset = temp_test_val.train_test_split(test_size=0.5, seed=42).values()

print(f"Train dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(validation_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")

In [ ]:
print(train_dataset)
print(test_dataset)

In [ ]:
from transformers import AutoTokenizer

model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"

tokenizer = AutoTokenizer.from_pretrained(
    model_name
)

print(tokenizer)

In [ ]:
sample = train_dataset[0]["messages"]
sample

In [ ]:
formatted = tokenizer.apply_chat_template(
    sample,
    tokenize=False,
    add_generation_prompt=False
)

print(formatted)

In [ ]:
tokens = tokenizer(
    formatted,
    add_specioal_tokens=False
)

print(f"Token count: {len(tokens['input_ids'])}")

In [ ]:
def token_lenght(example):
  text = tokenizer.apply_chat_template(
      example["messages"],
      tokenize=False,
      add_generation_prompt=False
  )

  return {
      "token_length": len(tokenizer(text)["input_ids"])
  }


In [ ]:
train_lenghts = train_dataset.map(
    token_lenght,
    remove_columns=train_dataset.column_names
)

test_lenghts = test_dataset.map(
    token_lenght,
    desc="Calculationg token lengths"
)

In [ ]:
length_df = pd.DataFrame(
    {
        "token_length": train_lenghts["token_length"]
    }
)

length_df["token_length"].describe()

In [ ]:
length_df["token_length"].plot(
    kind="hist",
    bins=50,
    figsize=(10, 5),
    title="Distribution of token lengths"
)
plt.xlabel("Tokens")
plt.show()

In [ ]:
print("Percentiles: ")
print(
    length_df["token_length"].quantile(
        [0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]
    )
)

In [ ]:
limits = [512, 768, 1024, 1536, 2048, 4096]

for limit in limits:
  percentage = (
      length_df["token_length"].le(limit).mean() * 100
  )

  print(f"{limit:4d} tokens > {percentage:2f}%")

In [ ]:
MAX_LENGTH = 1024

def tokenize_function(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )

    tokenized = tokenizer(
        text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False
    )

    return tokenized

In [ ]:
tokenized_train = train_dataset.map(
    tokenize_function,
    remove_columns=train_dataset.column_names,
    desc="Tokenizing train dataset"
)

tokenized_val = validation_dataset.map(
    tokenize_function,
    remove_columns=validation_dataset.column_names, # Corrected from train_dataset.column_names
    desc="Tokenizing validation dataset" # Changed description
)

tokenized_test = test_dataset.map(
    tokenize_function,
    remove_columns=test_dataset.column_names,
    desc="Tokenizing test dataset"
)

In [ ]:
print(tokenized_train)
print(tokenized_train[0].keys())

In [ ]:
import torch

print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:",
      round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
      "GB")

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
import torch
from transformers import (
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    AutoTokenizer
)

model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

In [ ]:
print(f"Device: {model.device}")

In [ ]:
from peft import prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

In [ ]:
from peft import LoraConfig

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ],
)



In [ ]:
model.add_adapter(peft_config)


In [ ]:
from trl import SFTConfig

training_args = SFTConfig(
    output_dir="/content/uzbek-deepseek-lora",

    # Sequence
    max_length=1024,

    # Batch
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,

    # Training
    num_train_epochs=2,
    learning_rate=2e-4,
    warmup_steps=142,

    # Optimization
    optim="paged_adamw_8bit",

    # Memory
    gradient_checkpointing=True,

    # Precision
    fp16=False, # Changed from True to False
    bf16=False, # Explicitly set to False

    # Evaluation
    eval_strategy="steps",
    eval_steps=500,

    # Saving
    save_strategy="steps",
    save_steps=250,
    save_total_limit=2,

    # Logging
    logging_steps=10,
    logging_first_step=True,

    # Dataset
    packing=False,

    # Important
    assistant_only_loss=False,

    report_to="none",
)

In [ ]:
print(tokenizer.chat_template)

In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train, # Changed from train_dataset
    eval_dataset=tokenized_val, # Changed from validation_dataset

    processing_class=tokenizer,

    peft_config=peft_config
)

In [ ]:
train_result = trainer.train()